# Capítulo 3: IA para la Detección de Amenazas

## 3.1. Introducción

La detección de amenazas es una de las aplicaciones más efectivas de la IA en seguridad. Los modelos de IA pueden analizar tráfico de red, comportamiento de usuarios y registros de sistema para identificar patrones anómalos que indiquen una posible brecha de seguridad.

En este notebook implementaremos:
- **Carga y exploración** de datos de tráfico de red.
- **Preprocesamiento y normalización** de las características.
- **Detección de anomalías** con Isolation Forest (ML clásico).
- **Detección de intrusiones** con Autoencoder (Deep Learning).

### Generación de datos simulados

Dado que el dataset real aún no está disponible, se genera un dataset sintético de tráfico de red que replica la estructura esperada. Esto permite desarrollar y probar todo el pipeline sin depender del archivo `network_traffic.csv`.

> **Nota:** Cuando se disponga del dataset real, basta con colocar el archivo `network_traffic.csv` en la carpeta `data/` y omitir esta celda.

In [ ]:
import numpy as np
import pandas as pd
import os

np.random.seed(42)
n_normal = 5000   # muestras de tráfico normal
n_anomaly = 250   # muestras anómalas (~5%)

# --- Tráfico normal ---
normal = pd.DataFrame({
    'bytes_sent':  np.random.exponential(500, n_normal).astype(int),
    'bytes_recv':  np.random.exponential(800, n_normal).astype(int),
    'duration':    np.random.exponential(30, n_normal).round(2),
    'protocol':    np.random.choice(['TCP', 'UDP', 'ICMP'], n_normal, p=[0.7, 0.25, 0.05]),
    'src_port':    np.random.randint(1024, 65535, n_normal),
    'dst_port':    np.random.choice([80, 443, 53, 22, 8080], n_normal),
    'label':       0   # 0 = normal
})

# --- Tráfico anómalo (valores extremos / patrones inusuales) ---
anomaly = pd.DataFrame({
    'bytes_sent':  np.random.exponential(5000, n_anomaly).astype(int),
    'bytes_recv':  np.random.exponential(50, n_anomaly).astype(int),
    'duration':    np.random.exponential(200, n_anomaly).round(2),
    'protocol':    np.random.choice(['TCP', 'UDP', 'ICMP'], n_anomaly, p=[0.3, 0.3, 0.4]),
    'src_port':    np.random.randint(1, 1024, n_anomaly),
    'dst_port':    np.random.choice([4444, 31337, 6667, 9999], n_anomaly),
    'label':       1   # 1 = anomalía
})

# Combinar y mezclar
data_sim = pd.concat([normal, anomaly], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

# Guardar en la carpeta data/
os.makedirs('data', exist_ok=True)
data_sim.to_csv('data/network_traffic.csv', index=False)
print(f"[OK] Dataset simulado guardado: data/network_traffic.csv ({len(data_sim)} registros)")

---
## 3.2. Recolección y preprocesamiento de datos

### 3.2.1. Carga de datos de tráfico de red

In [ ]:
# Listing 3.1: Carga y exploración de datos de tráfico de red

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar datos de tráfico de red
# El dataset debe incluir columnas como: bytes_sent, bytes_recv,
# duration, protocol, src_port, dst_port, etc.
data = pd.read_csv('data/network_traffic.csv')

print("=== Información del dataset ===")
print(data.info())
print("\n=== Primeras filas ===")
print(data.head())
print("\n=== Estadísticas descriptivas ===")
print(data.describe())

# Verificar valores nulos
print("\n=== Valores nulos por columna ===")
print(data.isnull().sum())

### 3.2.2. Preprocesamiento y normalización

In [ ]:
# Listing 3.2: Preprocesamiento completo del dataset de tráfico

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import matplotlib.pyplot as plt

# Cargar datos
data = pd.read_csv('data/network_traffic.csv')

# 1. Eliminar filas con valores nulos
data = data.dropna()

# 2. Codificar variables categóricas (e.g., protocolo)
le = LabelEncoder()
if 'protocol' in data.columns:
    data['protocol'] = le.fit_transform(data['protocol'])

# 3. Seleccionar características numéricas relevantes
feature_cols = ['bytes_sent', 'bytes_recv', 'duration',
                'src_port', 'dst_port', 'protocol']
features = data[feature_cols].copy()

# 4. Normalizar entre 0 y 1
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(features)
scaled_df = pd.DataFrame(scaled_data, columns=feature_cols)

print("Datos escalados (primeras 5 filas):")
print(scaled_df.head())

# 5. Visualizar distribución de bytes enviados
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(features['bytes_sent'], bins=50, color='steelblue', alpha=0.7)
plt.title('Distribución original - bytes_sent')
plt.subplot(1, 2, 2)
plt.hist(scaled_df['bytes_sent'], bins=50, color='darkorange', alpha=0.7)
plt.title('Distribución normalizada - bytes_sent')
plt.tight_layout()
plt.savefig('data/network_distribution.png', dpi=150)
plt.show()

---
## 3.3. Detección de anomalías con Isolation Forest

El algoritmo **Isolation Forest** aísla anomalías particionando el espacio de características de manera recursiva. Las anomalías son puntos que se aíslan con pocas particiones, lo que los hace fáciles de separar del resto.

In [ ]:
# Listing 3.3: Modelo de detección de anomalías con Isolation Forest

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report

# 1. Cargar y preparar datos
data = pd.read_csv('data/network_traffic.csv').dropna()

# Codificar protocolo para que sea numérico
le = LabelEncoder()
if data['protocol'].dtype == object:
    data['protocol'] = le.fit_transform(data['protocol'])

feature_cols = ['bytes_sent', 'bytes_recv', 'duration', 'src_port']
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data[feature_cols])

# 2. Entrenar Isolation Forest
# contamination: fracción esperada de anomalías (5%)
model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
    n_jobs=-1  # usar todos los núcleos disponibles
)
model.fit(scaled_data)

# 3. Predicción
#  1 = normal
# -1 = anomalía
predictions = model.predict(scaled_data)
scores = model.decision_function(scaled_data)  # puntuación de anomalía

data['anomaly'] = predictions
data['anomaly_score'] = scores

# 4. Resultados
anomalous = data[data['anomaly'] == -1]
normal = data[data['anomaly'] == 1]

print(f"Total de muestras: {len(data)}")
print(f"Muestras normales: {len(normal)}")
print(f"Anomalías detectadas: {len(anomalous)}")
print("\nTop 10 anomalías (menor puntuación = más anómalo):")
print(anomalous.sort_values('anomaly_score').head(10))

# 5. Evaluación contra etiquetas reales (si existen)
if 'label' in data.columns:
    # Convertir predicciones de IF (-1/1) a formato binario (1/0)
    y_pred = (predictions == -1).astype(int)
    y_true = data['label']
    print("\n=== Reporte de Clasificación ===")
    print(classification_report(y_true, y_pred, target_names=['Normal', 'Anomalía']))

# 6. Visualización
plt.figure(figsize=(10, 5))
plt.scatter(range(len(normal)),
            normal['bytes_sent'], s=5, c='steelblue',
            alpha=0.4, label='Normal')
plt.scatter(anomalous.index,
            anomalous['bytes_sent'], s=20, c='red',
            alpha=0.8, label='Anomalía')
plt.xlabel('Índice de muestra')
plt.ylabel('bytes_sent')
plt.title('Anomalías detectadas en tráfico de red')
plt.legend()
plt.tight_layout()
plt.savefig('data/anomaly_detection.png', dpi=150)
plt.show()

---
## 3.4. Detección de intrusiones con Autoencoder (Deep Learning)

Un **autoencoder** aprende a reconstruir entradas normales. Cuando procesa datos anómalos, el error de reconstrucción es alto, permitiendo detectar intrusiones no vistas durante el entrenamiento.

In [ ]:
# Listing 3.4: Autoencoder para detección de intrusiones

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 1. Preparar datos (solo tráfico normal para entrenamiento)
data = pd.read_csv('data/network_traffic.csv').dropna()

# Codificar protocolo
le = LabelEncoder()
if data['protocol'].dtype == object:
    data['protocol'] = le.fit_transform(data['protocol'])

feature_cols = ['bytes_sent', 'bytes_recv', 'duration', 'src_port']

scaler = MinMaxScaler()
X_all = scaler.fit_transform(data[feature_cols])
labels = data['label'].values

# Separar datos normales (label=0) para entrenar el autoencoder
X_normal = X_all[labels == 0]

# Dividir: 80% entrenamiento (solo normales), 20% validación
X_train, X_val = train_test_split(X_normal, test_size=0.2, random_state=42)

# El set de prueba incluye normales y anomalías para evaluar
X_test = X_all
y_test = labels

print(f"Muestras de entrenamiento (normales): {len(X_train)}")
print(f"Muestras de validación (normales):    {len(X_val)}")
print(f"Muestras de prueba (todas):           {len(X_test)}")

In [ ]:
# 2. Definir arquitectura del autoencoder
input_dim = X_train.shape[1]

def build_autoencoder(input_dim, encoding_dim=8):
    """Construye un autoencoder simétrico."""
    # Codificador
    inp = keras.Input(shape=(input_dim,))
    encoded = layers.Dense(16, activation='relu')(inp)
    encoded = layers.Dense(encoding_dim, activation='relu')(encoded)
    # Decodificador
    decoded = layers.Dense(16, activation='relu')(encoded)
    decoded = layers.Dense(input_dim, activation='sigmoid')(decoded)

    autoencoder = keras.Model(inp, decoded, name='autoencoder')
    encoder = keras.Model(inp, encoded, name='encoder')
    return autoencoder, encoder

autoencoder, encoder = build_autoencoder(input_dim)
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

In [ ]:
# 3. Entrenar
history = autoencoder.fit(
    X_train, X_train,
    epochs=50,
    batch_size=64,
    validation_data=(X_val, X_val),
    verbose=1
)

# 4. Calcular error de reconstrucción
X_test_pred = autoencoder.predict(X_test)
mse_err = np.mean(np.power(X_test - X_test_pred, 2), axis=1)

# Umbral: percentil 95 del error en datos de entrenamiento
X_train_pred = autoencoder.predict(X_train)
train_mse = np.mean(np.power(X_train - X_train_pred, 2), axis=1)
threshold = np.percentile(train_mse, 95)
print(f"Umbral de detección: {threshold:.6f}")

# Clasificar anomalías
anomalies_detected = (mse_err > threshold).astype(int)
print(f"Anomalías detectadas en test: {anomalies_detected.sum()} de {len(anomalies_detected)}")

# Evaluación contra etiquetas reales
from sklearn.metrics import classification_report
print("\n=== Reporte de Clasificación (Autoencoder) ===")
print(classification_report(y_test, anomalies_detected, target_names=['Normal', 'Anomalía']))

In [ ]:
# 5. Curva de pérdida
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Pérdida entrenamiento')
plt.plot(history.history['val_loss'], label='Pérdida validación')
plt.title('Curva de aprendizaje del Autoencoder')
plt.xlabel('Época')
plt.ylabel('MSE')
plt.legend()
plt.tight_layout()
plt.savefig('data/autoencoder_loss.png', dpi=150)
plt.show()

# 6. Distribución del error de reconstrucción
plt.figure(figsize=(8, 4))
plt.hist(mse_err[y_test == 0], bins=50, alpha=0.6, color='steelblue', label='Normal')
plt.hist(mse_err[y_test == 1], bins=50, alpha=0.6, color='red', label='Anomalía')
plt.axvline(threshold, color='black', linestyle='--', label=f'Umbral ({threshold:.4f})')
plt.title('Distribución del error de reconstrucción')
plt.xlabel('MSE')
plt.ylabel('Frecuencia')
plt.legend()
plt.tight_layout()
plt.savefig('data/autoencoder_reconstruction_error.png', dpi=150)
plt.show()

---
## 3.5. Guardado de artefactos

Se guardan el modelo de Isolation Forest y el Autoencoder para reutilizarlos en notebooks posteriores.

In [ ]:
import joblib
import os

os.makedirs('models', exist_ok=True)

# Guardar Isolation Forest
joblib.dump(model, 'models/isolation_forest.pkl')
print("[OK] Modelo Isolation Forest guardado en models/isolation_forest.pkl")

# Guardar Autoencoder
autoencoder.save('models/autoencoder_ids.keras')
print("[OK] Autoencoder guardado en models/autoencoder_ids.keras")

# Guardar el scaler para reutilizarlo
joblib.dump(scaler, 'models/scaler_network.pkl')
print("[OK] Scaler guardado en models/scaler_network.pkl")

# Guardar dataset con predicciones
data['anomaly_if'] = (model.predict(scaler.transform(data[feature_cols])) == -1).astype(int)
data['anomaly_ae'] = anomalies_detected
data.to_csv('data/network_traffic_predictions.csv', index=False)
print("[OK] Dataset con predicciones guardado en data/network_traffic_predictions.csv")